# Variant D — e-SNLI Explanation Generation

Generates **2,001 natural language explanations** from e-SNLI training data  
using `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B` via the HuggingFace Inference API.

Explanations follow the **e-SNLI paper methodology** (Camburu et al., NeurIPS 2018):
- Self-contained (readable without the premise/hypothesis)
- Focus on the non-obvious elements that determine the label
- Label-specific constraints per Section 3 of the paper
- Maximum 50 words
- Filtered against the uninformative templates from Appendix A

| | |
|---|---|
| **Target** | 667 per label × 3 = 2,001 traces (balanced) |
| **Rate limit** | HF free tier ~500–1,000 req/day → 2–3 daily sessions |
| **Timeout-safe** | Every row saved to Drive immediately after generation |
| **Resume** | Re-run notebook any time — skips completed rows automatically |

---
**Steps:**
1. Get a free HF token at https://huggingface.co/settings/tokens
2. Edit **Cell 2** only — paste your token and set your Drive path
3. Click **Runtime → Run all**
4. If rate limit hits mid-session, notebook prints instructions and stops cleanly
5. Next day: open notebook → **Run all** → resumes from last saved row

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q requests pandas

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Cell 2: CONFIGURATION  ←  Only cell you need to edit
# ══════════════════════════════════════════════════════════════════════

# 1. Your HuggingFace API token
#    Free token at: https://huggingface.co/settings/tokens
HF_TOKEN = "hf_PASTE_YOUR_TOKEN_HERE"

# 2. Path to your project folder on Google Drive
#    This folder must contain:  Datasets/E-SNLI/esnli_train_1.csv
#                               Datasets/E-SNLI/esnli_train_2.csv
PROJECT_ROOT = "/content/drive/MyDrive/Old-Explanations-New-Rationales-Bridging-e-SNLI-and-LLM-Chain-of-Thought-in-Encoder-Based-NLI"

# ══════════════════════════════════════════════════════════════════════
#  Do not edit below this line
# ══════════════════════════════════════════════════════════════════════

import os

MODEL_ID    = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
N_PER_LABEL = 667   # 667 × 3 = 2,001 total

ESNLI_TRAIN_1 = os.path.join(PROJECT_ROOT, "Datasets", "E-SNLI", "esnli_train_1.csv")
ESNLI_TRAIN_2 = os.path.join(PROJECT_ROOT, "Datasets", "E-SNLI", "esnli_train_2.csv")
SUBSET_PATH   = os.path.join(PROJECT_ROOT, "Datasets", "CoT", "cot_subset.csv")
OUTPUT_PATH   = os.path.join(PROJECT_ROOT, "Datasets", "CoT", "cot_traces.csv")

assert HF_TOKEN != "hf_PASTE_YOUR_TOKEN_HERE", (
    "Please paste your HuggingFace token in HF_TOKEN above.\n"
    "Get one free at: https://huggingface.co/settings/tokens"
)

print(f"Model:       {MODEL_ID}")
print(f"Target:      {N_PER_LABEL * 3:,} traces ({N_PER_LABEL} per label)")
print(f"e-SNLI:      {ESNLI_TRAIN_1}")
print(f"Subset:      {SUBSET_PATH}")
print(f"Output:      {OUTPUT_PATH}")
print(f"Token:       {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

import os
for p in [ESNLI_TRAIN_1, ESNLI_TRAIN_2]:
    exists = os.path.exists(p)
    status = "✓" if exists else "✗  NOT FOUND"
    print(f"{status}  {p}")

if not all(os.path.exists(p) for p in [ESNLI_TRAIN_1, ESNLI_TRAIN_2]):
    raise FileNotFoundError(
        "e-SNLI CSV files not found. Check that PROJECT_ROOT in Cell 2 "
        "points to the correct folder on your Drive."
    )

In [ ]:
# ── Cell 4: All logic (standalone — no project imports needed) ────────────

import re
import time
from pathlib import Path
from typing import Optional, Set

import pandas as pd
import requests

# ─────────────────────────────────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────────────────────────────────

HF_API_URL   = "https://api-inference.huggingface.co/v1/chat/completions"
VALID_LABELS = {"entailment", "neutral", "contradiction"}
MAX_WORDS    = 50

# ─────────────────────────────────────────────────────────────────────────
# Label-specific prompts — based on e-SNLI paper (Camburu et al. 2018)
#
# Section 3 methodology:
#   Entailment  → justify all hypothesis parts not in premise
#   Neutral     → state what hypothesis claims that cannot be inferred
#   Contradiction → state the specific conflicting elements from both sentences
#
# Appendix A banned templates are listed in the rules so the model avoids them.
# ─────────────────────────────────────────────────────────────────────────

PROMPTS = {
    "entailment": """\
You are an NLI explanation writer. The hypothesis FOLLOWS from the premise.
Write a SHORT explanation (max {max_words} words) of WHY the premise supports the hypothesis.

Premise:    {premise}
Hypothesis: {hypothesis}

Rules:
- Focus on the specific words or facts in the PREMISE that imply the hypothesis.
- Cover elements of the hypothesis that are NOT already stated in the premise.
- Write a self-contained sentence that makes sense without re-reading the inputs.
- Do NOT use the words entailment, neutral, contradiction.
- Do NOT use these banned templates: \"X implies Y\", \"If X then Y\",
  \"X is a rephrasing of Y\", \"X is the same as Y\", \"X is a synonym of Y\".

Good example: \"Man leans over a pickup truck implies that he is touching it.\"

Explanation ({max_words} words max):""",

    "neutral": """\
You are an NLI explanation writer. The hypothesis is NEITHER confirmed NOR contradicted by the premise.
Write a SHORT explanation (max {max_words} words) of WHAT the hypothesis claims that cannot be inferred.

Premise:    {premise}
Hypothesis: {hypothesis}

Rules:
- Focus on the specific concept in the HYPOTHESIS that the premise does not confirm.
- State the information gap directly — name the specific word or fact that is missing.
- Write a self-contained sentence that makes sense without re-reading the inputs.
- Do NOT use the words entailment, neutral, contradiction.
- Do NOT use these banned templates: \"Just because X doesn't mean Y\",
  \"Cannot infer Y\", \"One cannot assume Y\", \"We don't know that Y\",
  \"The fact that X doesn't mean Y\".

Good example: \"Child does not imply daughter and woman does not imply mother.\"

Explanation ({max_words} words max):""",

    "contradiction": """\
You are an NLI explanation writer. The hypothesis CONTRADICTS the premise.
Write a SHORT explanation (max {max_words} words) of WHAT specifically conflicts.

Premise:    {premise}
Hypothesis: {hypothesis}

Rules:
- Identify the specific element(s) from BOTH the premise AND the hypothesis that clash.
- State the conflict directly — refer to actual words or facts from both sentences.
- Write a self-contained sentence that makes sense without re-reading the inputs.
- Do NOT use the words entailment, neutral, contradiction.
- Do NOT use these banned templates: \"In sentence 1 X while in sentence 2 Y\",
  \"Either X or Y\", \"It can either be X or Y\", \"X contradicts Y\",
  \"X is contradictory to Y\", \"X is not the same as Y\".

Good example: \"Holds a stick implies using hands so it is not empty-handed.\"

Explanation ({max_words} words max):""",
}


def build_prompt(premise: str, hypothesis: str, label: str) -> str:
    return PROMPTS[label].format(
        premise=premise,
        hypothesis=hypothesis,
        max_words=MAX_WORDS,
    )


# ─────────────────────────────────────────────────────────────────────────
# Banned template detection (Appendix A of the paper)
# ─────────────────────────────────────────────────────────────────────────

# These regex patterns match the uninformative templates the paper filtered out.
# If a generated explanation matches, we retry once.
_BANNED = [
    r"just because .+ doesn.t mean",
    r"just because .+ does not",
    r"cannot infer",
    r"one cannot (assume|infer)",
    r"cannot assume",
    r"we don.t know that",
    r"the fact that .+ doesn.t mean",
    r"the fact that .+ does not (mean|imply|always)",
    r"^if .+then ",
    r" is a rephrasing of ",
    r" is the same as ",
    r" is a synonym of ",
    r"in both sentences",
    r"in sentence 1 .+ (while|and) in sentence 2",
    r"^(either|it can either be)",
    r" contradicts ",
    r" is contradictory to ",
    r"^it cannot be .+ if ",
    r" not the same as ",
]
_BANNED_RE = [re.compile(p, re.IGNORECASE) for p in _BANNED]


def is_banned(text: str) -> bool:
    """Return True if the text matches any Appendix-A banned template."""
    return any(r.search(text) for r in _BANNED_RE)


# ─────────────────────────────────────────────────────────────────────────
# Post-processing helpers
# ─────────────────────────────────────────────────────────────────────────

def strip_think_tags(text: str) -> str:
    """Remove <think>...</think> blocks produced by DeepSeek-R1 internally."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def truncate_to_words(text: str, max_words: int = MAX_WORDS) -> str:
    """Trim to max_words, preferring a clean sentence boundary."""
    words = text.split()
    if len(words) <= max_words:
        return text
    truncated = " ".join(words[:max_words])
    # Try to end at a sentence boundary
    for punct in (". ", "! ", "? "):
        idx = truncated.rfind(punct)
        if idx > len(truncated) // 3:   # don't chop more than 2/3 of the text
            return truncated[: idx + 1]
    return truncated


def clean(text: str) -> Optional[str]:
    """Strip think tags, truncate, reject if too short."""
    text = strip_think_tags(text)
    text = truncate_to_words(text)
    return text if len(text.split()) >= 3 else None


# ─────────────────────────────────────────────────────────────────────────
# Custom exception for clean rate-limit handling
# ─────────────────────────────────────────────────────────────────────────

class RateLimitExhausted(Exception):
    pass


# ─────────────────────────────────────────────────────────────────────────
# HuggingFace Inference API caller
# ─────────────────────────────────────────────────────────────────────────

def _call_api(prompt: str, max_retries: int = 5, timeout: int = 60) -> Optional[str]:
    """Single API call with retry logic.

    - 200  → return generated text
    - 429  → sleep Retry-After seconds, retry; raise RateLimitExhausted after max_retries
    - 503  → model loading, sleep 30s, retry
    - else → return None
    """
    headers = {
        "Authorization": f"Bearer {HF_TOKEN}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 120,     # ~90 words — enough for 50-word answer with overhead
        "temperature": 0.3,
        "stream": False,
    }

    rl_hits = 0
    for attempt in range(max_retries):
        try:
            resp = requests.post(HF_API_URL, headers=headers, json=payload, timeout=timeout)
        except requests.exceptions.Timeout:
            print(f"    [timeout attempt {attempt + 1}]", flush=True)
            time.sleep(10)
            continue
        except requests.exceptions.RequestException as e:
            print(f"    [connection error: {e}]", flush=True)
            return None

        if resp.status_code == 200:
            try:
                return resp.json()["choices"][0]["message"]["content"]
            except (KeyError, IndexError, ValueError):
                return None

        elif resp.status_code == 429:
            rl_hits += 1
            wait = int(resp.headers.get("Retry-After", 60))
            print(f"\n    [429 rate limit — waiting {wait}s ({rl_hits}/{max_retries})]", flush=True)
            time.sleep(wait)
            if rl_hits >= max_retries:
                raise RateLimitExhausted()

        elif resp.status_code == 503:
            print(f"\n    [503 model loading — waiting 30s]", flush=True)
            time.sleep(30)

        else:
            print(f"\n    [HTTP {resp.status_code}: {resp.text[:100]}]", flush=True)
            return None

    return None


def call_hf(premise: str, hypothesis: str, label: str) -> Optional[str]:
    """Generate one explanation. Retries once if output matches a banned template."""
    prompt = build_prompt(premise, hypothesis, label)
    raw    = _call_api(prompt)
    if raw is None:
        return None

    result = clean(raw)
    if result is None:
        return None

    # Retry once if banned template detected
    if is_banned(result):
        raw2   = _call_api(prompt)
        result = clean(raw2) if raw2 else result   # fall back to first if retry fails

    return result


# ─────────────────────────────────────────────────────────────────────────
# Data loading
# ─────────────────────────────────────────────────────────────────────────

def load_esnli() -> pd.DataFrame:
    """Load both e-SNLI training CSVs from Drive."""
    dfs = []
    for path in (ESNLI_TRAIN_1, ESNLI_TRAIN_2):
        df = pd.read_csv(
            path,
            usecols=["gold_label", "Sentence1", "Sentence2", "Explanation_1"],
        )
        dfs.append(df)
    df = pd.concat(dfs, ignore_index=True)
    df = df[df["gold_label"].isin(VALID_LABELS)]
    return df.dropna(subset=["Sentence1", "Sentence2", "Explanation_1"]).reset_index(drop=True)


def load_or_create_subset() -> pd.DataFrame:
    """Load the locked-in subset from Drive, or create and save it on first run.

    Saving the subset ensures every session uses the exact same rows and
    pair_ids — no drift between sessions regardless of environment changes.
    """
    p = Path(SUBSET_PATH)
    p.parent.mkdir(parents=True, exist_ok=True)

    if p.exists():
        subset = pd.read_csv(p, dtype={"pair_id": int})
        print(f"✓ Loaded existing subset ({len(subset):,} rows) from:\n  {p}")
    else:
        print("First run — creating stratified subset from e-SNLI...")
        esnli = load_esnli()
        print(f"  Loaded {len(esnli):,} e-SNLI training examples")
        subset = (
            esnli
            .groupby("gold_label", group_keys=False)
            .apply(lambda g: g.sample(min(N_PER_LABEL, len(g)), random_state=42))
            .reset_index(drop=True)
        )
        subset.insert(0, "pair_id", range(len(subset)))
        subset.to_csv(p, index=False)
        print(f"✓ Subset saved to Drive:\n  {p}")

    print(f"\n  Label counts:")
    for lbl, cnt in subset["gold_label"].value_counts().sort_index().items():
        print(f"    {lbl:<15} {cnt:,}")
    return subset


# ─────────────────────────────────────────────────────────────────────────
# Generation loop — checkpoint/resume
# ─────────────────────────────────────────────────────────────────────────

def generate(source_df: pd.DataFrame, sleep_between: float = 1.5) -> None:
    """Generate explanations with full checkpoint/resume support.

    Colab timeout:
        Every row is written to Drive immediately after generation.
        Re-run the notebook — completed rows are automatically skipped.

    Rate limit:
        On persistent 429s, raises RateLimitExhausted and stops cleanly.
        No partial rows are written. Re-run tomorrow — those rows retry.
    """
    out = Path(OUTPUT_PATH)
    out.parent.mkdir(parents=True, exist_ok=True)

    # Identify completed pair_ids — only rows with actual text count
    done: Set[int] = set()
    if out.exists():
        existing = pd.read_csv(out, usecols=["pair_id", "cot_rationale"],
                               dtype={"pair_id": int})
        done = set(
            existing.loc[
                existing["cot_rationale"].notna() &
                (existing["cot_rationale"].str.strip() != ""),
                "pair_id",
            ]
        )

    total     = len(source_df)
    remaining = source_df[~source_df["pair_id"].isin(done)].reset_index(drop=True)
    done_count = len(done)

    print(f"  Total target:  {total:,}")
    print(f"  Already done:  {done_count:,}")
    print(f"  Remaining:     {len(remaining):,}")
    print(f"  Output file:   {out}")
    print()

    if remaining.empty:
        print("All traces already generated — run the Validate cell below.")
        return

    file_exists = out.exists()
    start_time  = time.time()

    try:
        for i, row in enumerate(remaining.itertuples(index=False)):
            result = call_hf(row.Sentence1, row.Sentence2, row.gold_label)

            # Write one row immediately — this is the checkpoint
            pd.DataFrame([{
                "pair_id":       row.pair_id,
                "gold_label":    row.gold_label,
                "Sentence1":     row.Sentence1,
                "Sentence2":     row.Sentence2,
                "Explanation_1": row.Explanation_1,
                "cot_rationale": result if result is not None else "",
                "cot_model":     MODEL_ID,
            }]).to_csv(out, mode="a", header=not file_exists, index=False)

            file_exists = True
            done_count += 1

            if (i + 1) % 50 == 0:
                elapsed = (time.time() - start_time) / 60
                rate    = (i + 1) / elapsed if elapsed > 0 else 0
                eta     = (total - done_count) / rate if rate > 0 else float("inf")
                print(
                    f"  [{done_count:,}/{total:,}]  "
                    f"elapsed={elapsed:.1f}min  "
                    f"rate={rate:.1f}/min  "
                    f"ETA≈{eta:.0f}min",
                    flush=True,
                )

            time.sleep(sleep_between)

    except RateLimitExhausted:
        elapsed = (time.time() - start_time) / 60
        new_this_session = done_count - len(done)
        print()
        print("━" * 58)
        print("  DAILY RATE LIMIT REACHED")
        print("━" * 58)
        print(f"  Generated this session:  {new_this_session:,} traces")
        print(f"  Total completed so far:  {done_count:,} / {total:,}")
        print(f"  Still remaining:         {total - done_count:,}")
        print(f"  Session time:            {elapsed:.1f} min")
        print()
        print("  All progress is saved to Drive. Nothing is lost.")
        print("  ─────────────────────────────────────────────────")
        print("  NEXT STEPS:")
        print("  Come back tomorrow, open this notebook,")
        print("  and click  Runtime → Run all.")
        print("  Generation resumes automatically from this point.")
        print("━" * 58)
        return

    elapsed_total = (time.time() - start_time) / 60
    print(f"\n✓ Complete! {done_count:,}/{total:,} traces saved in {elapsed_total:.1f} min.")


# ─────────────────────────────────────────────────────────────────────────
# Validation report
# ─────────────────────────────────────────────────────────────────────────

def validate() -> None:
    p = Path(OUTPUT_PATH)
    if not p.exists():
        print(f"Output file not found: {p}")
        return

    df        = pd.read_csv(p, dtype={"pair_id": int})
    target    = N_PER_LABEL * 3
    non_empty = df["cot_rationale"].notna() & (df["cot_rationale"].str.strip() != "")
    filled    = non_empty.sum()

    leakage = df.loc[non_empty, "cot_rationale"].str.lower().apply(
        lambda t: any(w in t for w in ["entailment", "neutral", "contradiction"])
    ).sum()

    banned_count = df.loc[non_empty, "cot_rationale"].apply(is_banned).sum()

    wc = df.loc[non_empty, "cot_rationale"].str.split().apply(len)

    pct = filled / target * 100
    bar = "█" * int(pct / 2) + "░" * (50 - int(pct / 2))

    print("═" * 52)
    print("  Validation Report")
    print("═" * 52)
    print(f"  [{bar}]")
    print(f"  Progress:         {filled:>5,} / {target:,}  ({pct:.1f}%)")
    print(f"  Label leakage:    {leakage:>5,}  ({leakage/max(filled,1)*100:.1f}%)  ← target <5%")
    print(f"  Banned templates: {banned_count:>5,}  ({banned_count/max(filled,1)*100:.1f}%)  ← target <10%")
    print(f"  Avg word count:   {wc.mean():>5.1f}  (max allowed: {MAX_WORDS})")
    print(f"  Min / Max words:  {wc.min()} / {wc.max()}")
    print("  Label distribution:")
    for lbl, cnt in df["gold_label"].value_counts().sort_index().items():
        print(f"    {lbl:<15} {cnt:>5,}  ({cnt/len(df)*100:.1f}%)")
    print("═" * 52)
    if filled < target:
        print(f"  {target - filled:,} traces still needed.")
        print("  Run the Generate cell again tomorrow.")
    else:
        print("  ✓ Generation complete — file is ready for Variant D training.")


print("✓ All functions loaded.")

In [ ]:
# ── Cell 5: Load (or create) the locked-in subset ─────────────────────────
#
# First run:  reads both e-SNLI training CSVs from Drive,
#             selects 667 rows per label (2,001 total), saves cot_subset.csv.
#
# Every subsequent run:  loads cot_subset.csv — exact same rows every time.
#
# WHY: Saving the subset guarantees pair_ids are stable across sessions.
#      If only the output CSV existed, re-generating the subset could produce
#      different rows if the environment changes between sessions.

subset = load_or_create_subset()

In [ ]:
# ── Cell 6: Session status ─────────────────────────────────────────────────

from pathlib import Path
import pandas as pd

target = N_PER_LABEL * 3
out    = Path(OUTPUT_PATH)

print("─" * 45)
if out.exists():
    df_s  = pd.read_csv(out)
    done  = df_s["cot_rationale"].notna() & (df_s["cot_rationale"].str.strip() != "")
    n     = done.sum()
    pct   = n / target * 100
    bar   = "█" * int(pct / 2) + "░" * (50 - int(pct / 2))
    print(f"  [{bar}]  {pct:.1f}%")
    print(f"  Done:       {n:,} / {target:,}")
    print(f"  Remaining:  {target - n:,}")
    print()
    print("  Label breakdown (completed):")
    for lbl, cnt in df_s.loc[done, "gold_label"].value_counts().sort_index().items():
        print(f"    {lbl:<15} {cnt:,}")
else:
    print(f"  No output yet — {target:,} traces to generate.")
print("─" * 45)

In [ ]:
# ── Cell 7: API test — verify token and model before full run ──────────────

print("Testing API with one entailment example...")
test = call_hf(
    premise="A dog is running through a field.",
    hypothesis="An animal is outside.",
    label="entailment",
)

if test:
    print(f"✓ API working ({len(test.split())} words)")
    print("─" * 50)
    print(test)
    print("─" * 50)
    print(f"  Banned template? {'YES — will retry during generation' if is_banned(test) else 'No'}")
    print(f"  Label leakage?   {'YES' if any(w in test.lower() for w in ['entailment','neutral','contradiction']) else 'No'}")
else:
    print("✗ API test failed.")
    print(f"  Check token:  {HF_TOKEN[:8]}...")
    print(f"  Check model:  {MODEL_ID}")
    print("  Fix before running Cell 8.")

In [ ]:
# ── Cell 8: Generate ───────────────────────────────────────────────────────
#
# Safe to re-run at any time:
#
#   Colab timeout   → All rows written to Drive up to the last one.
#                     Re-run notebook → skips completed rows automatically.
#
#   Rate limit hit  → Stops cleanly with clear instructions.
#                     No incomplete rows written.
#                     Re-run tomorrow → retries remaining rows.
#
#   Already done    → Detects completion, tells you to validate.

generate(subset)

In [ ]:
# ── Cell 9: Validate ───────────────────────────────────────────────────────
validate()

In [ ]:
# ── Cell 10: Preview — one sample per label ────────────────────────────────
import pandas as pd

df_p     = pd.read_csv(OUTPUT_PATH)
done_p   = df_p["cot_rationale"].notna() & (df_p["cot_rationale"].str.strip() != "")
samples  = df_p[done_p].groupby("gold_label").head(1).reset_index(drop=True)

for _, row in samples.iterrows():
    print(f"Label:        {row['gold_label']}")
    print(f"Premise:      {row['Sentence1']}")
    print(f"Hypothesis:   {row['Sentence2']}")
    print(f"Human (e-SNLI): {row['Explanation_1']}")
    print(f"Generated:    {row['cot_rationale']}")
    print(f"Words:        {len(str(row['cot_rationale']).split())}")
    print("─" * 60)